# Poketeer Card Embedder — DINOv2

Generates card embeddings using **DINOv2** (Meta's vision foundation model) for accurate Pokemon card identification via visual similarity search.

No training needed — DINOv2 is pretrained on 142M images and already understands visual similarity.

**Runtime → Change runtime type → GPU (T4)** for faster embedding generation.

In [ ]:
# 1. Check GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — will still work, just slower')

In [ ]:
# 2. Download card catalog from Supabase
!pip install -q supabase

SUPABASE_URL = "https://bohtisqptjwbmlfcnliz.supabase.co"
# Anon key is fine here — cards table has public read via RLS
SUPABASE_ANON_KEY = "sb_publishable_ge1aLD1g3od3jjzBe8vTpQ_qvpB3Gfw"

from supabase import create_client
sb = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)

# Fetch all cards (paginated — Supabase limits to 1000 per request)
all_cards = []
page_size = 1000
offset = 0
while True:
    res = sb.table('cards').select('id, name, number, set_id, rarity, image_small, image_large, supertype, subtypes, hp, artist, types').range(offset, offset + page_size - 1).execute()
    batch = res.data or []
    all_cards.extend(batch)
    if len(batch) < page_size:
        break
    offset += page_size

print(f'Loaded {len(all_cards)} cards from Supabase')

In [ ]:
# 3. Download all card images (cached to disk)
import os
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

IMG_DIR = Path('card_images')
IMG_DIR.mkdir(exist_ok=True)

def download_image(card):
    url = card.get('image_small', '')
    if not url:
        return None
    path = IMG_DIR / f"{card['id']}.png"
    if path.exists():
        return str(path)
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        path.write_bytes(r.content)
        return str(path)
    except:
        return None

# Download with 16 threads
card_paths = {}
with ThreadPoolExecutor(max_workers=16) as pool:
    futures = {pool.submit(download_image, c): c['id'] for c in all_cards}
    for f in tqdm(as_completed(futures), total=len(futures), desc='Downloading'):
        card_id = futures[f]
        path = f.result()
        if path:
            card_paths[card_id] = path

print(f'Downloaded {len(card_paths)} / {len(all_cards)} card images')

In [ ]:
# 4. Load DINOv2 model
import torch.nn as nn
from torchvision import transforms

# DINOv2 ViT-B/14 — good balance of accuracy and speed
# Produces 768-D embeddings, we project to 512-D for pgvector
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dinov2 = dinov2.to(device)
dinov2.eval()

# Simple projection to match our 512-D pgvector column
projection = nn.Linear(768, 512, bias=False).to(device)
# Initialize with truncated SVD-style (preserve most variance)
nn.init.orthogonal_(projection.weight)

print(f'DINOv2 ViT-B/14 loaded on {device}')
print(f'Output: 768-D → projected to 512-D')

# Preprocessing for DINOv2 (must match its training)
eval_transform = transforms.Compose([
    transforms.Resize(252, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# 5. Generate embeddings for all cards
import numpy as np
import torch.nn.functional as F
from PIL import Image
from tqdm.notebook import tqdm

BATCH_SIZE = 64
all_embeddings = []
all_ids = []
valid_cards = [c for c in all_cards if c['id'] in card_paths]

print(f'Generating embeddings for {len(valid_cards)} cards...')

for i in tqdm(range(0, len(valid_cards), BATCH_SIZE), desc='Embedding'):
    batch_cards = valid_cards[i:i+BATCH_SIZE]
    tensors = []
    batch_ids = []
    for card in batch_cards:
        try:
            img = Image.open(card_paths[card['id']]).convert('RGB')
            tensors.append(eval_transform(img))
            batch_ids.append(card['id'])
            img.close()
        except:
            continue

    if not tensors:
        continue

    batch_tensor = torch.stack(tensors).to(device)
    with torch.inference_mode():
        # Get DINOv2 features
        features = dinov2(batch_tensor)  # (B, 768)
        # Project to 512-D and L2-normalize
        embs = projection(features)
        embs = F.normalize(embs, p=2, dim=1)
        all_embeddings.append(embs.cpu().numpy())
    all_ids.extend(batch_ids)

embeddings_matrix = np.concatenate(all_embeddings, axis=0)
print(f'\nDone! {len(all_ids)} embeddings, shape: {embeddings_matrix.shape}')

In [ ]:
# 6. Quick validation — test similarity matching
import random

# Pick a random card and check its top matches
test_idx = random.randint(0, len(all_ids) - 1)
test_id = all_ids[test_idx]
test_emb = embeddings_matrix[test_idx]
test_card = next(c for c in all_cards if c['id'] == test_id)

# Cosine similarity against all (embeddings are L2-normalized)
sims = embeddings_matrix @ test_emb
top_indices = np.argsort(sims)[::-1][:10]

print(f'Test card: {test_card["name"]} ({test_id})')
print(f'\nTop 10 matches:')
for rank, idx in enumerate(top_indices):
    match_id = all_ids[idx]
    match_card = next(c for c in all_cards if c['id'] == match_id)
    marker = ' ← SELF' if idx == test_idx else ''
    print(f'  {rank+1}. {sims[idx]:.4f}  {match_id:20s}  {match_card["name"]}{marker}')

# Score spread analysis
print(f'\nScore spread: top1={sims[top_indices[0]]:.4f}, top2={sims[top_indices[1]]:.4f}, top5={sims[top_indices[4]]:.4f}')
print(f'Gap (top1-top2): {sims[top_indices[0]] - sims[top_indices[1]]:.4f}')
print(f'Mean similarity: {sims.mean():.4f}, Std: {sims.std():.4f}')
print(f'\nA large gap between top1 and top2 means the model can clearly distinguish cards.')

In [ ]:
# 7. Save the projection weights (needed by HF Space)
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=False)
SAVE_DIR = Path('/content/drive/MyDrive/poketeer_training')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save just the projection layer (tiny — the DINOv2 backbone is loaded from torch.hub)
torch.save(projection.state_dict(), str(SAVE_DIR / 'dinov2_projection.pt'))
torch.save(projection.state_dict(), 'dinov2_projection.pt')

print(f'Projection saved to Google Drive and locally')
print(f'File size: {(SAVE_DIR / "dinov2_projection.pt").stat().st_size / 1024:.1f} KB')

In [ ]:
# 8. Upload embeddings to Supabase
SUPABASE_SERVICE_KEY = input('Paste your Supabase SERVICE KEY: ')

sb_admin = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

BATCH = 100
from tqdm.notebook import tqdm as tqdm2

for i in tqdm2(range(0, len(all_ids), BATCH), desc='Uploading embeddings'):
    rows = []
    for j in range(i, min(i + BATCH, len(all_ids))):
        vec = embeddings_matrix[j].tolist()
        vec_str = '[' + ','.join(f'{v:.6f}' for v in vec) + ']'
        rows.append({'card_id': all_ids[j], 'embedding': vec_str})
    try:
        sb_admin.table('card_embeddings').upsert(rows).execute()
    except Exception as e:
        print(f'\n  Batch {i} failed: {e}, retrying...')
        import time; time.sleep(2)
        sb_admin.table('card_embeddings').upsert(rows).execute()

print(f'\nUploaded {len(all_ids)} embeddings to Supabase')

In [ ]:
# 9. Download projection weights for HF Space
from google.colab import files
files.download('dinov2_projection.pt')
print('Download dinov2_projection.pt and upload it to your HF Space')
print('The DINOv2 backbone loads automatically from torch.hub — no large model file needed!')